In [29]:
import gradio as gr
from PyPDF2 import PdfReader
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex
from llama_cpp import Llama
from llama_index.core.vector_stores import MetadataFilters, FilterOperator, MetadataFilter
import json
import tempfile
import os

# 1. Initialize LLM and Embedding Model once at startup
llm = Llama.from_pretrained(
    repo_id="bartowski/Mistral-7B-Instruct-v0.3-GGUF",
    filename="Mistral-7B-Instruct-v0.3-Q5_K_S.gguf",
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False
)

gemini_model = lambda p: llm.create_chat_completion(
    messages=[{"role": "user", "content": p}],
    max_tokens=512,
    temperature=0.0
)["choices"][0]["message"]["content"]

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Global state to store the processed index and latest query output
index = None
last_results_dict = {}

VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

def clean_doc_type(response):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    cleaned_title = cleaned.title()
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label
    return cleaned_title

def is_same_document(prev_text, curr_text, doc_type=None):
    prompt = f"""You are comparing the full text of two consecutive pages that were both classified as '{doc_type}'.
Determine whether Current Page is a continuation of the exact same document from Previous Page, or a separate standalone document.

Read the full pages carefully and apply these content rules:
- Answer 'No' (separate documents) if both pages are Certificates of Quality for different Lot Numbers or different products.
- Answer 'No' (separate documents) if Current Page restarts at Section 1 or introduces a brand new standalone record.
- Answer 'Yes' (same document) ONLY if Current Page continues the unfinished sections, tables, or text of the exact same document from Previous Page (for example, Previous Page ends at Section 3 and Current Page starts at Section 4).

Previous Page Full Text:
{prev_text}

Current Page Full Text:
{curr_text}

Do these two pages belong to the SAME individual document? Answer ONLY 'Yes' or 'No'."""
    
    response = gemini_model(prompt)
    return response.strip().lower().startswith("yes")

def classify_document_type(text):
    prompt = f"""
    You are a pharmaceutical document classifier. Based on the page
    content below, classify it into ONE of these document types:

    - Cover Letter: A formal letter (often starting with "To Whom It
      May Concern") discussing product information or storage conditions.
    - Certificate of Quality: Contains lot numbers, manufacture dates,
      expiration dates, and test results (autoclave, gamma irradiation).
    - Packaging Specification: Describes packaging components, materials,
      part numbers, and configuration change history.
    - BSE/TSE Declaration: A declaration about animal-origin materials
      and transmissible spongiform encephalopathy compliance.
    - Material Description: Lists materials of construction, sterilization
      compatibility, and physical properties of a product.
    - Supplier Qualification: Contains supplier audit history,
      certifications (ISO 9001, ISO 13485), and approved product lists.
    - Chain of Custody: Lists manufactured assemblies, traceability
      information, and the manufacturing-to-shipment flow.
    - Other: Use ONLY if the content does not match any of the above.

    Page Content:
    {text}

    Respond with ONLY the document type name. No explanation.
    """
    response = gemini_model(prompt)
    return clean_doc_type(response)

def predict_doc_type_for_query(query):
    prompt = f"""
    You are an intelligent assistant that routes user queries to the most relevant pharmaceutical document.

    User query: "{query}"

    Which document type is most likely to contain the answer?
    Choose ONLY ONE from: Cover Letter, Certificate of Quality, Packaging Specification, BSE/TSE Declaration,
    Material Description, Supplier Qualification, Chain of Custody, Other.

    Respond with ONLY the document type name. No explanation.
    """
    response = gemini_model(prompt)
    return clean_doc_type(response)

# 2. Process the uploaded PDF and build the VectorStoreIndex
def process_pdf(file):
    global index
    if file is None:
        return "⚠️ Please upload a PDF file first."

    file_path = file.name if hasattr(file, "name") else file
    reader = PdfReader(file_path)
    pages = [page.extract_text() or "" for page in reader.pages]
    doc_pages = [{"page_num": i, "text": p} for i, p in enumerate(pages)]

    metadata_store = []
    page_counter = 0

    for i, page in enumerate(doc_pages):
        curr_doc_type = classify_document_type(page["text"])
        
        if i == 0:
            page_counter = 0
        else:
            prev_text = doc_pages[i - 1]["text"]
            prev_doc_type = metadata_store[i - 1]["doc_type"]
            
            if curr_doc_type == prev_doc_type:
                same = is_same_document(prev_text, page["text"], curr_doc_type)
            else:
                same = False
                
            if same:
                page_counter += 1
            else:
                page_counter = 0
    
        metadata_store.append({
            "page": i,
            "text": page["text"],
            "doc_type": curr_doc_type,
            "page_in_doc": page_counter
        })

    logical_docs = []
    current_doc = {"text": "", "doc_type": None, "page_start": 0}

    for page in metadata_store:
        if page["page_in_doc"] == 0 and current_doc["text"]:
            current_doc["page_end"] = page["page"] - 1
            logical_docs.append(current_doc)
            current_doc = {"text": "", "doc_type": None, "page_start": page["page"]}
    
        current_doc["text"] += "\n\n" + page["text"]
        current_doc["doc_type"] = page["doc_type"]
    
    if metadata_store:
        current_doc["page_end"] = metadata_store[-1]["page"]
        logical_docs.append(current_doc)

    chunked_documents = [
        Document(
            text=doc["text"].strip(),
            metadata={
                "doc_type": doc["doc_type"],
                "page_start": doc["page_start"],
                "page_end": doc["page_end"]
            }
        )
        for doc in logical_docs
    ]

    index = VectorStoreIndex.from_documents(chunked_documents, embed_model=embed_model)
    return f"✅ Processed {len(doc_pages)} pages into {len(chunked_documents)} logical documents."

# 3. Query the processed PDF when the user clicks Send
def handle_chat(message, history):
    global index, last_results_dict
    history = history or []

    if not message.strip():
        return history, ""

    if index is None:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": "⚠️ Please upload a PDF and click 'Process Document' first."})
        return history, ""

    predicted_doc_type = predict_doc_type_for_query(message)
    print(f"Predicted document type: {predicted_doc_type}")

    retriever = index.as_retriever(
        filters=MetadataFilters(
            filters=[
                MetadataFilter(key="doc_type", value=predicted_doc_type, operator=FilterOperator.EQ)
            ]
        )
    )
    results = retriever.retrieve(message)

    if not results:
        retriever = index.as_retriever()
        results = retriever.retrieve(message)

    if results:
        retrieved_text = "\n\n---\n\n".join([res.text for res in results])
        prompt = f"""Based on the following text, please answer the user's question.

Text:
{retrieved_text}

User Question: {message}

Provide a clear, specific answer based only on the text provided."""
        answer = gemini_model(prompt)
    else:
        answer = "Could not find relevant information in the uploaded document."

    # Build the Step 14 structured dictionary
    last_results_dict = {
        "query": message,
        "predicted_doc_type": predicted_doc_type,
        "matched_chunks": [
            {
                "text": res.text,
                "metadata": res.metadata
            }
            for res in results
        ],
        "answer": answer
    }

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})
    return history, ""

# ============================================================
# STEP 14: Structured Dictionary & Chat History Downloads
# ============================================================
def dict_out():
    """Writes the Step 14 dictionary to a JSON file and returns the file path for download."""
    output_path = os.path.join(tempfile.gettempdir(), "rag_results.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(last_results_dict, f, indent=2)
    return output_path

def download_chat_history(history):
    """Writes the current Gradio chat history to a JSON file and returns the file path for download."""
    output_path = os.path.join(tempfile.gettempdir(), "chat_history.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(history or [], f, indent=2)
    return output_path

# 4. Gradio UI
with gr.Blocks(title="Step 5: Full Functioning UI") as demo:

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat History", height=350)
            user_query = gr.Textbox(
                placeholder="Ask a question about your document...",
                label="Your Question"
            )
            with gr.Row():
                send_btn = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear Chat")

        with gr.Column(scale=1):
            pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
            process_btn = gr.Button("🔄 Process Document", variant="secondary")
            status_box = gr.Textbox(label="Status", interactive=False, value="Waiting for PDF...")
            
            gr.Markdown("---")
            with gr.Row():
                download_dict_btn = gr.Button("📥 Download Query Details and Metadata")
                download_chat_btn = gr.Button("💬 Download Chat History")
            download_file_output = gr.File(label="Download File", interactive=False)

    # Event Wiring
    process_btn.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=status_box
    )
    send_btn.click(
        fn=handle_chat,
        inputs=[user_query, chatbot],
        outputs=[chatbot, user_query]
    )
    user_query.submit(
        fn=handle_chat,
        inputs=[user_query, chatbot],
        outputs=[chatbot, user_query]
    )
    clear_btn.click(lambda: [], outputs=chatbot)
    
    # Generate and provide the downloadable files when clicked
    download_dict_btn.click(
        fn=dict_out,
        inputs=None,
        outputs=download_file_output
    )
    download_chat_btn.click(
        fn=download_chat_history,
        inputs=chatbot,
        outputs=download_file_output
    )

demo.launch()

/Users/leo/Pfizer-RAG-AI-Pipeline/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 14194.28it/s]


* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


Predicted document type: Chain Of Custody
